
# Interleaved vs Independent MLP Experiment

This notebook compares:

1. Two independently trained MLP models  
2. Two models trained with layer-wise random interleaving  
3. Hybrid inference behavior and variance  

It generates:
- Training curves
- Hybrid variance plots
- Stress test comparisons


In [ ]:

import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:

from dataclasses import dataclass

@dataclass
class TwoViewConfig:
    n_train: int = 8000
    n_test: int = 2000
    d_view: int = 8
    noise_std: float = 0.8
    view_noise: float = 0.5

def make_two_view_data(cfg: TwoViewConfig, seed: int):
    set_seed(seed)
    w = torch.randn(cfg.d_view)

    def sample(n):
        z = torch.randn(n, cfg.d_view)
        eps = cfg.noise_std * torch.randn(n)
        logits = (z @ w) + eps
        y = (logits > 0).long()
        xA = z + cfg.view_noise * torch.randn(n, cfg.d_view)
        xB = z + cfg.view_noise * torch.randn(n, cfg.d_view)
        x = torch.cat([xA, xB], dim=1)
        return x, y

    train = sample(cfg.n_train)
    test = sample(cfg.n_test)
    return train, test

def make_loaders(x_train, y_train, x_test, y_test, batch_size=256):
    train_ds = TensorDataset(x_train, y_train)
    test_ds = TensorDataset(x_test, y_test)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(test_ds, batch_size=batch_size)
    )


In [ ]:

class MLP(nn.Module):
    def __init__(self, d_in, hidden, depth):
        super().__init__()
        layers = []
        dims = [d_in] + [hidden] * depth
        for i in range(depth):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            layers.append(nn.ReLU())
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 2)

    def forward(self, x):
        return self.head(self.backbone(x))


In [ ]:

def forward_interleaved(x, modelA, modelB, mask):
    h = x
    depth = len(mask)
    for i in range(depth):
        bb = modelB.backbone if mask[i] else modelA.backbone
        h = bb[2*i](h)
        h = bb[2*i+1](h)
    return modelA.head(h)


In [ ]:

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    total_correct = 0
    total = 0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction="sum")
        total_loss += loss.item()
        total_correct += (logits.argmax(1)==y).sum().item()
        total += y.size(0)
    return total_loss/total, total_correct/total

def train(model, train_loader, test_loader, steps=300):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = []
    for step in range(steps):
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
            break
        if step % 50 == 0:
            nll, acc = evaluate(model, test_loader)
            history.append(acc)
    return history


In [ ]:

# Run experiment

cfg = TwoViewConfig()
(train_x, train_y), (test_x, test_y) = make_two_view_data(cfg, seed=42)
train_loader, test_loader = make_loaders(train_x, train_y, test_x, test_y)

d_in = 2 * cfg.d_view
depth = 4
hidden = 64

set_seed(1)
modelA = MLP(d_in, hidden, depth).to(device)
histA = train(modelA, train_loader, test_loader)

set_seed(2)
modelB = MLP(d_in, hidden, depth).to(device)
histB = train(modelB, train_loader, test_loader)

plt.figure()
plt.plot(histA)
plt.title("Independent Model A Accuracy")
plt.xlabel("Evaluation Step")
plt.ylabel("Accuracy")
plt.show()

plt.figure()
plt.plot(histB)
plt.title("Independent Model B Accuracy")
plt.xlabel("Evaluation Step")
plt.ylabel("Accuracy")
plt.show()
